In [2]:
import torch
import sys

print(f"Python Version: {sys.version}")
print(f"CUDA Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.2f} GB")
else:
    print("WARNING: GPU not detected. Check CUDA installation.")

Python Version: 3.13.13 (tags/v3.13.13:01104ce, Apr  7 2026, 19:25:48) [MSC v.1944 64 bit (AMD64)]
CUDA Available: True
GPU: NVIDIA GeForce RTX 4050 Laptop GPU
VRAM: 6.00 GB


In [3]:
from diffusers import AutoencoderKL

# 1. Clear any remaining cache
torch.cuda.empty_cache()

# 2. Load directly to GPU using device_map
# This is the most memory-efficient way to load a model
vae = AutoencoderKL.from_pretrained(
    "runwayml/stable-diffusion-v1-5", 
    subfolder="vae",
    torch_dtype=torch.float16,
    variant="fp16",
    device_map="auto" # This automatically places it on the GPU efficiently
)

print("VAE Loaded Successfully!")
print(f"VRAM used: {torch.cuda.memory_allocated(0) / 1024**2:.2f} MB")

d:\DLProject\venv\Lib\site-packages\huggingface_hub\utils\_validators.py:205: UserWarning: The `local_dir_use_symlinks` argument is deprecated and ignored in `hf_hub_download`. Downloading to a local directory does not use symlinks anymore.
  warnings.warn(


VAE Loaded Successfully!
VRAM used: 160.57 MB


In [4]:
from transformers import CLIPTextModel, CLIPTokenizer

# Tokenizer doesn't use GPU memory, it stays on CPU
tokenizer = CLIPTokenizer.from_pretrained(
    "runwayml/stable-diffusion-v1-5", 
    subfolder="tokenizer"
)

# Text Encoder is the 'Conditioning' (y) part of the paper
text_encoder = CLIPTextModel.from_pretrained(
    "runwayml/stable-diffusion-v1-5", 
    subfolder="text_encoder",
    torch_dtype=torch.float16,
    device_map="auto"
)

print("Text Encoder Loaded!")
print(f"Total VRAM used: {torch.cuda.memory_allocated(0) / 1024**2:.2f} MB")

Loading weights: 100%|██████████| 196/196 [00:00<00:00, 1492.30it/s]


Text Encoder Loaded!
Total VRAM used: 398.54 MB


In [5]:
from diffusers import UNet2DConditionModel

unet = UNet2DConditionModel.from_pretrained(
    "runwayml/stable-diffusion-v1-5", 
    subfolder="unet",
    torch_dtype=torch.float16,
    variant="fp16",
    device_map="auto"
)

print("U-Net Loaded Successfully!")
print(f"Total VRAM used: {torch.cuda.memory_allocated(0) / 1024**2:.2f} MB")

U-Net Loaded Successfully!
Total VRAM used: 2047.66 MB


In [6]:
from diffusers import LMSDiscreteScheduler

# This matches the 'Linear Noise Schedule' mentioned in the paper
scheduler = LMSDiscreteScheduler.from_pretrained(
    "runwayml/stable-diffusion-v1-5", 
    subfolder="scheduler"
)

print("Scheduler defined.")

Scheduler defined.


In [12]:
from tqdm.auto import tqdm
from PIL import Image
import numpy as np
import math

image_index = math.floor(10 * np.random.rand())  # Random index for saving the image
# 1. Configuration - Using CPU for the random generator to avoid CUDA unknown errors
prompt = ["A high-tech signal processing research lab at IIT Jammu"]
height, width = 512, 512
steps = 50 
guidance_scale = 7.5  
# Create generator on CPU
generator = torch.Generator(device="cpu").manual_seed(42) 

# 2. Encode the text prompt
text_input = tokenizer(prompt, padding="max_length", max_length=tokenizer.model_max_length, truncation=True, return_tensors="pt")
with torch.no_grad():
    # Only move the specific tensor needed for the encoder to 'cuda'
    text_embeddings = text_encoder(text_input.input_ids.to("cuda"))[0]

# 3. Prepare Unconditional Embeddings
max_length = text_input.input_ids.shape[-1]
uncond_input = tokenizer([""], padding="max_length", max_length=max_length, return_tensors="pt")
with torch.no_grad():
    uncond_embeddings = text_encoder(uncond_input.input_ids.to("cuda"))[0]

text_embeddings = torch.cat([uncond_embeddings, text_embeddings])

# 4. Create initial random noise in LATENT space (64x64)
# We create it on CPU first then move it to GPU
latents = torch.randn(
    (1, unet.config.in_channels, height // 8, width // 8), 
    generator=generator
).to("cuda", dtype=torch.float16)

latents = latents * scheduler.init_noise_sigma

print("Setup complete. Ready for Denoising Loop.")

# 5. Denoising Loop
scheduler.set_timesteps(steps)

for t in tqdm(scheduler.timesteps):
    # Expand latents for classifier-free guidance
    latent_model_input = torch.cat([latents] * 2)
    latent_model_input = scheduler.scale_model_input(latent_model_input, t)

    # Predict noise residual
    with torch.no_grad():
        noise_pred = unet(latent_model_input, t, encoder_hidden_states=text_embeddings).sample

    # Perform guidance
    noise_pred_uncond, noise_pred_text = noise_pred.chunk(2)
    noise_pred = noise_pred_uncond + guidance_scale * (noise_pred_text - noise_pred_uncond)

    # Compute previous noisy sample (z_t -> z_t-1)
    latents = scheduler.step(noise_pred, t, latents).prev_sample

# 6. Decode Latent -> Image
latents = 1 / 0.18215 * latents # 0.18215 is the scaling factor from the paper
with torch.no_grad():
    image = vae.decode(latents).sample

# 7. Convert to PIL
image = (image / 2 + 0.5).clamp(0, 1)
image = image.detach().cpu().permute(0, 2, 3, 1).numpy()
image = (image * 255).astype("uint8")
final_image = Image.fromarray(image[0])
final_image.save(f"Generated_Image/result_{image_index}.png")
final_image.show()

print(f"Final VRAM used: {torch.cuda.memory_allocated(0) / 1024**2:.2f} MB")

Setup complete. Ready for Denoising Loop.


100%|██████████| 50/50 [00:05<00:00,  8.57it/s]


Final VRAM used: 2059.59 MB


In [15]:
import torch
import numpy as np
#import requests         # We won't use this since we're loading from a local path
#from io import BytesIO  # We won't use this since we're loading from a local path
from skimage.metrics import peak_signal_noise_ratio as psnr_metric

random_index = math.floor(10 * np.random.rand())  # Random index for saving the image

# 1. Loading a test image
local_path = "D:/DLProject/Images/image_2.jpg"

try:
    original_pil = Image.open(local_path).convert("RGB").resize((512, 512))
    print("Local image loaded successfully!")
except FileNotFoundError:
    print(f"Error: Could not find the image at {local_path}. Please check the folder.")

### In case we want to load from a URL instead, uncomment the following lines and provide a valid URL
# url = ""  # Replace with your local path or a valid URL
# response = requests.get(url)
# original_pil = Image.open(BytesIO(response.content)).convert("RGB").resize((512, 512))

# 2. Pre-process for VAE (Scale to -1 to 1)
img_array = np.array(original_pil).astype(np.float32) / 127.5 - 1.0
img_tensor = torch.from_numpy(img_array).permute(2, 0, 1).unsqueeze(0).to("cuda", dtype=torch.float16)

# 3. The Test: Encode -> Decode
with torch.no_grad():
    # Phase A: Spatial Compression (E)
    posterior = vae.encode(img_tensor).latent_dist
    latents = posterior.mode() # We use mode for a deterministic reconstruction
    
    # Phase B: Reconstruction (D)
    reconstruction = vae.decode(latents).sample

# 4. Post-process back to Pixels
recon_array = (reconstruction.squeeze().cpu().permute(1, 2, 0).numpy() + 1.0) * 127.5
recon_array = recon_array.clip(0, 255).astype(np.uint8)
recon_pil = Image.fromarray(recon_array)

# 5. Calculate Metrics for your "Results Section"
local_psnr = psnr_metric(np.array(original_pil), recon_array)
paper_psnr = 27.1  # Benchmark from the LDM paper for f=8
delta_percent = ((local_psnr - paper_psnr) / paper_psnr) * 100

print(f"--- RECONSTRUCTION TEST RESULTS ---")
print(f"Local PSNR: {local_psnr:.2f} dB")
print(f"Paper PSNR: {paper_psnr} dB")
print(f"Delta: {delta_percent:.2f}%")

# Save the side-by-side comparison for your Qualitative Results slide
comparison = Image.new('RGB', (1024, 512))
comparison.paste(original_pil, (0, 0))
comparison.paste(recon_pil, (512, 0))
comparison.save(f"Images&ReconstructedImages/reconstruction_comparison_{random_index}.png")
comparison.show()

Error: Could not find the image at D:/DLProject/Images/image_2.jpg. Please check the folder.
--- RECONSTRUCTION TEST RESULTS ---
Local PSNR: 25.28 dB
Paper PSNR: 27.1 dB
Delta: -6.72%
